In [ ]:
%pip install catboost
from catboost.datasets import amazon
train_df, test_df = amazon()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 8.9 MB/s eta 0:00:00


In [ ]:
#  Basic Overview
print("Shape:", train_df.shape)
print("Columns:", train_df.columns.tolist())
train_df.head()


#  Basic Info
print(train_df.info())
print("\nDescriptive Stats:\n", train_df.describe())
print("\nMissing Values:\n", train_df.isnull().sum())

Shape: (32769, 10)
Columns: ['ACTION', 'RESOURCE', 'MGR_ID', 'ROLE_ROLLUP_1', 'ROLE_ROLLUP_2', 'ROLE_DEPTNAME', 'ROLE_TITLE', 'ROLE_FAMILY_DESC', 'ROLE_FAMILY', 'ROLE_CODE']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32769 entries, 0 to 32768
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   ACTION            32769 non-null  int64
 1   RESOURCE          32769 non-null  int64
 2   MGR_ID            32769 non-null  int64
 3   ROLE_ROLLUP_1     32769 non-null  int64
 4   ROLE_ROLLUP_2     32769 non-null  int64
 5   ROLE_DEPTNAME     32769 non-null  int64
 6   ROLE_TITLE        32769 non-null  int64
 7   ROLE_FAMILY_DESC  32769 non-null  int64
 8   ROLE_FAMILY       32769 non-null  int64
 9   ROLE_CODE         32769 non-null  int64
dtypes: int64(10)
memory usage: 2.5 MB
None

Descriptive Stats:
              ACTION       RESOURCE         MGR_ID  ROLE_ROLLUP_1  \
count  32769.000000   32769.000000   32769.0

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.base import BaseEstimator, TransformerMixin

In [ ]:
class Winsorizer(BaseEstimator, TransformerMixin):
    def __init__(self, lower=0.01, upper=0.99):
        self.lower = lower
        self.upper = upper
        self.feature_names_in_ = None

    def fit(self, X, y=None):
        if isinstance(X, pd.DataFrame):
            self.feature_names_in_ = X.columns
            X = X.values
        self.lower_bounds_ = np.percentile(X, self.lower * 100, axis=0)
        self.upper_bounds_ = np.percentile(X, self.upper * 100, axis=0)
        return self

    def transform(self, X):
        is_df = isinstance(X, pd.DataFrame)
        if is_df:
            cols = X.columns
            X = X.values

        X_transformed = np.copy(X)
        for i in range(X.shape[1]):
            X_transformed[:, i][X_transformed[:, i] < self.lower_bounds_[i]] = self.lower_bounds_[i]
            X_transformed[:, i][X_transformed[:, i] > self.upper_bounds_[i]] = self.upper_bounds_[i]

        return pd.DataFrame(X_transformed, columns=cols) if is_df else X_transformed

    def get_feature_names_out(self, input_features=None):
        if input_features is None and self.feature_names_in_ is not None:
            return self.feature_names_in_
        elif input_features is not None:
            return input_features
        else:
            # Fallback if fit was not called with a DataFrame
            return [f"x{i}" for i in range(self.lower_bounds_.shape[0])]

In [ ]:
X = train_df.drop(columns=['ACTION'])
y = train_df['ACTION']

# Identify columns
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include='object').columns.tolist()

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

In [ ]:
# Numerical pipeline
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('winsor', Winsorizer()),
    ('scaler', StandardScaler())
])

# Categorical pipeline
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

In [ ]:
models = {
    "XGBoost": XGBClassifier(n_estimators=248, max_depth=10, learning_rate=0.14577436504812752, subsample=0.7194066455990469, colsample_bytree=0.6635829320595318, gamma=0.25906123138975756, use_label_encoder=False, eval_metric='logloss')
}

pipelines = {}
for name, model in models.items():
    pipelines[name] = ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('smote', SMOTE(random_state=42)),
        ('model', model)
    ])

In [ ]:
results = {}
for name, pipe in pipelines.items():
    print(f"Evaluating {name}...")
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_pred_proba = pipe.predict_proba(X_test)[:, 1]

    report = classification_report(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)

    results[name] = {
        "classification_report": report,
        "roc_auc_score": roc_auc
    }

    print(f"--- {name} ---")
    print(report)
    print(f"ROC AUC: {roc_auc:.4f}")
    print("-" * 20)

Evaluating XGBoost...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:183: UserWarning: [18:03:02] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


--- XGBoost ---
              precision    recall  f1-score   support

           0       0.49      0.40      0.44       379
           1       0.96      0.97      0.97      6175

    accuracy                           0.94      6554
   macro avg       0.73      0.69      0.71      6554
weighted avg       0.94      0.94      0.94      6554

ROC AUC: 0.8451
--------------------


In [ ]:
import pickle

# Access your pipeline
model_pipeline = pipelines["XGBoost"]

# Now pickle it safely
with open('model_pipeline.pkl', 'wb') as f:
    pickle.dump(model_pipeline, f, protocol=4)

In [ ]:
# Cell 1: Create requirements.txt
%%writefile requirements.txt
streamlit
pandas
numpy
scikit-learn
xgboost
imbalanced-learn

Writing requirements.txt


In [ ]:
%%writefile app.py

import streamlit as st
import pickle
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
# Import ImbPipeline from imblearn is necessary for unpickling the full pipeline object
from imblearn.pipeline import Pipeline as ImbPipeline

# --- CRITICAL: Redefine the custom Winsorizer class ---
# The deployment environment (Streamlit) must know what this class is
# before it can successfully load the pickled object.
class Winsorizer(BaseEstimator, TransformerMixin):
    """
    Custom transformer to cap outliers using percentile-based winsorization.
    Matches the class definition used during model training in the notebook.
    """
    def __init__(self, lower=0.01, upper=0.99):
        self.lower = lower
        self.upper = upper
        self.feature_names_in_ = None

    def fit(self, X, y=None):
        if isinstance(X, pd.DataFrame):
            self.feature_names_in_ = X.columns
            X = X.values
        # Calculate the lower and upper bounds for each feature
        self.lower_bounds_ = np.percentile(X, self.lower * 100, axis=0)
        self.upper_bounds_ = np.percentile(X, self.upper * 100, axis=0)
        return self

    def transform(self, X):
        is_df = isinstance(X, pd.DataFrame)
        if is_df:
            # Preserve column names if input was a DataFrame
            cols = X.columns
            X = X.values

        X_transformed = np.copy(X)

        # Apply winsorization: values below the lower bound are set to the lower bound,
        # and similarly for the upper bound.
        for i in range(X.shape[1]):
            X_transformed[:, i][X_transformed[:, i] < self.lower_bounds_[i]] = self.lower_bounds_[i]
            X_transformed[:, i][X_transformed[:, i] > self.upper_bounds_[i]] = self.upper_bounds_[i]

        return pd.DataFrame(X_transformed, columns=cols) if is_df else X_transformed

    def get_feature_names_out(self, input_features=None):
        """Returns the feature names, essential for compatibility with ColumnTransformer."""
        if input_features is None and self.feature_names_in_ is not None:
            return self.feature_names_in_
        elif input_features is not None:
            return input_features
        else:
            # Fallback if fit was not called with a DataFrame
            return [f"x{i}" for i in range(self.lower_bounds_.shape[0])]
# -------------------------------------------------------------


# List of feature names (must match the order and names used during training)
FEATURE_NAMES = [
    'RESOURCE', 'MGR_ID', 'ROLE_ROLLUP_1', 'ROLE_ROLLUP_2',
    'ROLE_DEPTNAME', 'ROLE_TITLE', 'ROLE_FAMILY_DESC',
    'ROLE_FAMILY', 'ROLE_CODE'
]

# Function to load the model pipeline using Streamlit's cache
@st.cache_resource
def load_model():
    """Loads the pickled model and caches it to prevent reloading on every interaction."""
    try:
        # Assumes the file is named 'model_pipeline.pkl' and is in the same directory
        with open('model_pipeline.pkl', 'rb') as f:
            pipeline = pickle.load(f)
        return pipeline
    except FileNotFoundError:
        st.error("🚨 Deployment Error: Model file 'model_pipeline.pkl' not found. Ensure it is uploaded to your GitHub repository alongside app.py.")
        return None

# Load the model
model_pipeline = load_model()

# --- Streamlit App Structure ---
st.set_page_config(page_title="Access Predictor", layout="wide")
st.title("Amazon Employee Access Prediction 🤖")
st.markdown("""
    Enter employee role data (all numerical IDs) to predict the **ACTION** required:
    **1 (Approve)** or **0 (Deny)**.
""")

if model_pipeline:

    # Input section organized in a form for atomic submission
    with st.form("input_form", clear_on_submit=False):

        st.header("Enter Role IDs")
        input_data = {}

        # Layout features across three columns for clean UI
        col1, col2, col3 = st.columns(3)

        # Dynamic creation of input fields
        for i, feature in enumerate(FEATURE_NAMES):
            current_col = [col1, col2, col3][i % 3]

            # Use number_input since all features are categorical IDs represented as integers
            input_data[feature] = current_col.number_input(
                f"🔢 {feature}",
                min_value=0,
                value=10000, # Providing a typical large default ID
                step=1,
                key=feature
            )

        st.markdown("---")
        # Submission button
        submit_button = st.form_submit_button("Get Prediction", type="primary")

    # Prediction logic
    if submit_button:
        try:
            # 1. Convert input dictionary to a DataFrame
            # The column names and order must exactly match the training data
            input_df = pd.DataFrame([input_data])

            # 2. Make Prediction (pipeline handles Winsorizer, scaling, SMOTE, and XGBoost)
            prediction = model_pipeline.predict(input_df)[0]

            # Get probability for the positive class (Action=1)
            prediction_proba_1 = model_pipeline.predict_proba(input_df)[0, 1]

            st.subheader("Final Access Decision")
            st.code(f"Predicted Probability of Approval (Action=1): {prediction_proba_1:.4f}")

            if prediction == 1:
                st.success(f"🎉 Access **APPROVED**! The system predicts this request should be granted.")
            else:
                st.error(f"🚫 Access **DENIED**. The system recommends against granting this request.")

        except Exception as e:
            st.error(f"An error occurred during prediction. Please check the input values. Error: {e}")
else:
    st.info("Waiting for the model pipeline to load successfully.")

Overwriting app.py
